In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

INPUT_DIM = 28 * 28   # 784 пикселя
HIDDEN_DIM1 = 256     # 1-й скрытый слой
HIDDEN_DIM2 = 128     # 2-й скрытый слой
OUTPUT_DIM = 10       # 10 классов (цифры 0-9)

class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super().__init__()

        # 3 полносвязных слоя
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)

    def forward(self, x):
        # разворачиваем картинку в вектор (batch_size, 784)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def predict_proba(self, x):
        """Функция для получения вероятностей (Softmax)."""
        logits = self.forward(x)
        probs = F.softmax(logits, dim=1)
        return probs


In [ ]:
# Трансформация: перевод в тензор [0, 1]
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
device = torch.device("cpu")

model = SimpleNet(INPUT_DIM, HIDDEN_DIM1, HIDDEN_DIM2, OUTPUT_DIM).to(device)

criterion = nn.CrossEntropyLoss()                 # Внутри сама делает Softmax по логитам
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
EPOCHS = 3 

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Обнуляем градиенты
        optimizer.zero_grad()

        # Прямой проход
        outputs = model(images)     

        # Считаем loss
        loss = criterion(outputs, labels)

        # Обратный проход
        loss.backward()

        # Шаг оптимизации
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Эпоха {epoch+1}/{EPOCHS}, средний loss: {avg_loss:.4f}")


Эпоха 1/3, средний loss: 0.2840
Эпоха 2/3, средний loss: 0.1079
Эпоха 3/3, средний loss: 0.0723


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)               
        _, predicted = torch.max(outputs, 1)  # индекс максимума по классу

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total * 100
print(f"Точность на тесте: {accuracy:.2f}%")


Точность на тесте: 97.49%


In [ ]:
for images, labels in test_loader:
    break  

# Обрабатываем первые 10 картинок
for i in range(10):
    img = images[i:i+1].to(device)          # батч из 1 картинки
    true_label = labels[i].item()           # правильная цифра
    
    probs = model.predict_proba(img)        # вероятности
    pred = probs.argmax(dim=1).item()       # предсказанный класс
    
    print(f"Картинка {i}: истина = {true_label}, предсказано = {pred}")
    print("Вероятности:", probs.cpu().detach().numpy())
    print("-" * 50)


Картинка 0: истина = 7, предсказано = 7
Вероятности: [[1.5582525e-07 5.2036944e-07 3.6583301e-06 8.2681026e-06 4.5634736e-09
  9.3915112e-08 1.8184489e-11 9.9998224e-01 4.6141065e-07 4.6679952e-06]]
--------------------------------------------------
Картинка 1: истина = 2, предсказано = 2
Вероятности: [[2.1217250e-08 3.5896462e-03 9.9605006e-01 3.1199807e-04 2.3852909e-09
  1.1596125e-05 7.4031753e-08 3.2150183e-08 3.6590387e-05 2.1766705e-10]]
--------------------------------------------------
Картинка 2: истина = 1, предсказано = 1
Вероятности: [[4.8268885e-06 9.9369872e-01 2.7886432e-04 2.9623487e-05 5.9448148e-04
  3.7130278e-05 2.1417382e-05 4.6185129e-03 6.8847835e-04 2.7930841e-05]]
--------------------------------------------------
Картинка 3: истина = 0, предсказано = 0
Вероятности: [[9.9741167e-01 9.3779889e-08 2.5528012e-04 7.4874811e-06 2.4662324e-05
  4.3946961e-06 1.8668883e-05 1.0000523e-03 1.3814493e-05 1.2639128e-03]]
--------------------------------------------------
